# Spatial Copilot — Kaggle notebook (fetch → dataset → train → evaluate)

Everything needed to train on Kaggle in one file. Same code as `main.py` / `scripts/*` —
this notebook only orchestrates it.

**Before you run:** in *Settings → Internet* turn **ON** (needed for `git clone`, pip and the
HuggingFace dataset). *Accelerator:* **GPU T4 ×2** (or P100).

| Step | Cell | Notes |
|---|---|---|
| Fetch repo | 1 | clones into `/kaggle/working/Spatial_model` |
| Install deps | 2 | `ultralytics` + dataset tooling; Kaggle's CUDA torch is left alone |
| Build dataset | 3 | CubiCasa5K from HuggingFace → `/kaggle/temp/yolo` (~1.5 GB) |
| Train | 4 | YOLO11-seg with per-epoch progress bars |
| Evaluate | 5 | per-class IoU (floor / wall / door / window) |
| Load / infer | 6 | segmenter + trained YOLO on one plan |
| Persist | 7 | copy weights where Kaggle keeps them |
| Serve (optional) | 8 | API + viewer through an ngrok HTTPS tunnel |

Kaggle wipes `/kaggle/temp` and stops the session after **9 h on GPU**; only `/kaggle/working`
(20 GB output limit) survives. Hence: dataset → `temp`, weights → `working`.

---
## 1 — Fetch the repo

In [ ]:
# @title Clone / locate the repo + import the loader
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/joshyajith863/Spatial_model.git"
IN_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if IN_KAGGLE else Path.cwd()
REPO = WORK / "Spatial_model"
PULL = True   # re-sync if the repo was cloned by an earlier session

if (Path.cwd() / "main.py").exists():          # notebook already inside the repo
    REPO = Path.cwd()
elif not (REPO / "main.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
elif PULL:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)

os.chdir(REPO)
sys.path.insert(0, str(REPO))
import main as loader  # noqa: E402

print("repo  :", REPO)
print("kaggle:", IN_KAGGLE)
print("commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                 capture_output=True, text=True).stdout.strip())
try:
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found (no NVIDIA GPU on this machine)")

---
## 2 — Install training dependencies

Kaggle already ships a CUDA build of torch, so install only what's missing. Never
`pip install -r requirements.txt` here — it pins CPU torch.

In [ ]:
INSTALL = True

if INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                    "ultralytics", "pandas", "pyarrow", "huggingface_hub", "pyyaml"],
                   check=False)

import torch, ultralytics
print("torch      :", torch.__version__, "cuda:", torch.cuda.is_available())
print("ultralytics:", ultralytics.__version__)
if torch.cuda.is_available():
    print("device     :", torch.cuda.get_device_name(0))

---
## 3 — Build the dataset (CubiCasa5K → YOLO-seg)

Downloads the Parquet/CUBiCasa5K COCO export from HuggingFace and writes
`images/`, `labels/`, `data.yaml`. Goes to `/kaggle/temp` so ~10k files don't eat the
20 GB *output* budget.

Already have a Kaggle **dataset** with the converted files attached? Set
`ATTACHED_DATA_YAML` and skip the download.

In [ ]:
DATA_ROOT = Path("/kaggle/temp/yolo") if IN_KAGGLE else Path("data/yolo")
ATTACHED_DATA_YAML = None      # e.g. "/kaggle/input/cubicasa-yolo/data.yaml"
REBUILD = False                # True to re-download even if the yaml exists

if ATTACHED_DATA_YAML:
    DATA_YAML = Path(ATTACHED_DATA_YAML)
elif (DATA_ROOT / "data.yaml").exists() and not REBUILD:
    DATA_YAML = DATA_ROOT / "data.yaml"
    print("reusing existing dataset:", DATA_YAML)
else:
    subprocess.run([sys.executable, "scripts/convert_cubicasa_yolo.py",
                    "--split", "both", "--out", str(DATA_ROOT)], check=True)
    DATA_YAML = DATA_ROOT / "data.yaml"

counts = {s: len(list((DATA_ROOT / "images" / d).glob("*")))
          for s, d in (("train", "train"), ("val", "valid"))}
print("data.yaml:", DATA_YAML)
print("images   :", counts)

---
## 4 — Train

`LIMIT` builds a real subset on disk (Ultralytics has no image cap) — start with 200 for a
few-minute smoke run, then set `LIMIT = 0` for the full 4,228-image split. Weights are written
to `/kaggle/working/runs`, which survives the session.

In [ ]:
EPOCHS = 2            # 50+ for a real run
BATCH = 16            # T4 x2 / P100 handle 16 at 640; use 8 if you see OOM
IMGSZ = 640
DEVICE = "0"          # Kaggle gives "0" and "1"
LIMIT = 200           # 0 = all 4,228 train images
NAME = "cubicasa_kaggle"
PROJECT = str(WORK / "runs" / "segment")
WORKERS = 4
RESUME = None         # e.g. str(WORK / "runs/segment/cubicasa_kaggle/weights/last.pt")

data_arg = loader.build_subset_dataset(DATA_YAML, LIMIT) if LIMIT else DATA_YAML

loader.train_model(data=Path(data_arg), epochs=EPOCHS, batch=BATCH, imgsz=IMGSZ,
                   device=DEVICE, project=PROJECT, name=NAME, workers=WORKERS,
                   resume=RESUME, exist_ok=True)

---
## 5 — Evaluate per class

Runs `scripts/eval_yolo.py` on the validation split and prints the IoU table.

In [ ]:
WEIGHTS = WORK / "runs/segment" / NAME / "weights/best.pt"
print("weights:", WEIGHTS, "exists:", WEIGHTS.exists())

if WEIGHTS.exists():
    subprocess.run([sys.executable, "scripts/eval_yolo.py",
                    "--weights", str(WEIGHTS), "--data", str(DATA_YAML)], check=False)

---
## 6 — Load the trained model on a plan

Needs the segmentation-model deps; the cell installs them if missing. `image=None` picks the
first validation plan.

In [ ]:
INSTALL_APP = True
if INSTALL_APP:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "opencv-python-headless", "safetensors",
                    "segmentation-models-pytorch", "pymupdf"], check=False)

IMAGE = None
if IMAGE is None:
    plans = sorted((DATA_ROOT / "images" / "valid").glob("*.png"))
    IMAGE = str(plans[0]) if plans else None
print("image:", IMAGE)

loader.load_yolo(weights=WEIGHTS if WEIGHTS.exists() else None, image=IMAGE, device=DEVICE)

---
## 7 — Keep the weights

`/kaggle/working` is saved as the notebook **output** (20 GB, and commit waits for it).
Copy just the checkpoint(s) into a small folder, then use *File → Save version* and either
download them or publish the output as a Kaggle dataset.

In [ ]:
import shutil

ARTIFACTS = WORK / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
run_dir = WORK / "runs/segment" / NAME / "weights"

for name in ("best.pt", "last.pt"):
    src = run_dir / name
    if src.exists():
        shutil.copy2(src, ARTIFACTS / f"{NAME}_{name}")
        print(f"saved {ARTIFACTS / (NAME + '_' + name)}  ({src.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"missing {src}")

print("\noutput tree:")
for p in sorted(ARTIFACTS.iterdir()):
    print(" ", p.name, f"{p.stat().st_size/1e6:.1f} MB")

---
## 8 (optional) — Serve the API + viewer through ngrok

Kaggle exposes no port proxy, so a tunnel is needed. ngrok gives **HTTPS**, which is also what
WebXR requires in a headset. Needs a free authtoken from dashboard.ngrok.com.

In [ ]:
RUN_SERVER = False
NGROK_TOKEN = ""      # paste your token, then set RUN_SERVER = True

if RUN_SERVER:
    import threading
    import time

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok",
                    "fastapi", "uvicorn", "shapely", "trimesh", "mapbox-earcut",
                    "ezdxf"], check=False)
    threading.Thread(target=loader.serve, kwargs={"host": "0.0.0.0", "port": 8000},
                     daemon=True).start()
    time.sleep(3)

    from pyngrok import ngrok
    if NGROK_TOKEN:
        ngrok.set_auth_token(NGROK_TOKEN)
    tunnel = ngrok.connect(8000, "http")
    print("docs  :", tunnel.public_url + "/docs")
    print("viewer:", tunnel.public_url + "/static/viewer.html")

---
## Time limits & resume

- GPU sessions stop after **9 h** (12 h CPU). Kaggle does not auto-resume a notebook.
- Rerun cells 1–3 (the dataset lives in `/kaggle/temp`, so it is re-downloaded), then set
  `RESUME` in cell 4 to `/kaggle/working/runs/segment/cubicasa_kaggle/weights/last.pt`.
- `--patience` (default 20) already early-stops a run that has stopped improving.
- CPU-only throughput in this repo is ~1 h/epoch; keep `DEVICE="0"` and a GPU accelerator.

| Notebook | CLI |
|---|---|
| cell 3 | `python scripts/convert_cubicasa_yolo.py --split both --out /kaggle/temp/yolo` |
| cell 4 | `python main.py train --device 0 --epochs 50 --batch 16` |
| cell 5 | `python scripts/eval_yolo.py --weights .../best.pt` |
| cell 6 | `python main.py load yolo --weights .../best.pt` |